In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import bidirectional_dataset
import anomaly_dataset
import two_way_seq2seq
import linear_benchmark
import trainer
import plotting
import neptune
from sklearn.metrics import confusion_matrix

# Anomaly detection
We train the seq2seq model on subjects ["001", "002", "004", "005", "007", "009"]
and calculate the prediction MSE for all sections of subject "003". This is done both for the linear benchmark model and for the seq2seqmodel.
The 95th percentile of the prediction MSE for each model is calculated, and this is used as a threshold - all observations
with MSE above this are considered anomalous.
Finally, this anomaly detection rule is tested on subject "008" on data where 25% of sections have had
anomalies inserted.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_proportion_to_use = 1
validation_proportion_to_use = 1

# in seconds
time_before_cutout = 1
cutout_duration = 1
time_after_cutout = 1

# defining the resampling stuff
original_freq = 200
resample_freq = 20

n_channels = 7
cutout_duration_steps = cutout_duration * resample_freq
n_epoch = 60
epochs_with_teacher_forcing = 3

# HYPERPARAMETERS

batch_size = 256
hidden_size = 1024
lr = 0.001
initial_teacher_forcing = 0.2
encoder_dropout = 0.2
num_layers = 3

# Training Model 

In [ ]:
# Selecting subjects
training_subjects = ["001", "002", "004", "005", "007", "009"]
validation_subjects = ["003"]

# log the run on neptiune
run = neptune.init_run(
    project="sleep-time-series/sleep-time-series",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJlNjYwOTA1Zi02ZDhiLTQ5NGYtODUwYy1jNzA3ZmM0MjBhMWEifQ==",
    name= "Anomaly detection",
    tags=["Anomaly detection"]
    )

run["params"] = {
    "batch_size": batch_size,
    "hidden_size": hidden_size,
    "learning_rate": lr,
    "initial_teacher_forcing": initial_teacher_forcing,
    "encoder_droput": encoder_dropout,
    "n_epoch": n_epoch,
    "epochs_with_teacher_forcing": epochs_with_teacher_forcing,
    "num_layers": num_layers
    } 

run["training_subjects"] = ", ".join(training_subjects)
run["validation_subjects"] = ", ".join(validation_subjects)

# create datasets
combined_train_dataset, individual_train_datasets, standardization_info = bidirectional_dataset.create_combined_dataset(subjects=training_subjects,
                                                                                                 proportion_to_use=1, 
                                                                                                 time_before_cutout=time_before_cutout,
                                                                                                 cutout_duration=cutout_duration,
                                                                                                 original_freq=original_freq, 
                                                                                                 resample_freq=resample_freq)

# Create a combined and individual validation dataset, standardize using training set
combined_validation_dataset, individual_validation_datasets, _ = bidirectional_dataset.create_combined_dataset(subjects=validation_subjects,
                                                                                proportion_to_use=1, 
                                                                                time_before_cutout=time_before_cutout,
                                                                                cutout_duration=cutout_duration,
                                                                                original_freq=original_freq, 
                                                                                resample_freq=resample_freq,
                                                                                standardization_info=standardization_info)

# Create dataloaders
train_loader = torch.utils.data.DataLoader(combined_train_dataset, batch_size=batch_size, shuffle=True)

validation_loader = torch.utils.data.DataLoader(combined_validation_dataset, batch_size=batch_size, shuffle=False)

# Setup the model
pred_model = two_way_seq2seq.TwoWaySeq2Seq(hidden_size, n_channels, cutout_duration_steps, encoder_dropout=encoder_dropout, num_layers=num_layers)
pred_model = nn.DataParallel(pred_model, device_ids = [0, 1])
pred_model.to(device)
optimizer = torch.optim.Adam(params = pred_model.parameters(), lr = lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = 3)
loss_fn = nn.MSELoss(reduction="mean")

# Train the model
trainer.pred_training_loop(pred_model, n_epoch, loss_fn, optimizer, train_loader, validation_loader, 
                           neptune_run=run,
                           device=device, scheduler=scheduler, 
                           initial_teacher_forcing=initial_teacher_forcing, 
                           epochs_with_teacher_forcing=epochs_with_teacher_forcing)

# Plots for neptune
plot1 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 40, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
plot2 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 200, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)
plot3 = plotting.plot_target_and_prediction(pred_model, individual_validation_datasets, 600, 0, device,
                                   time_before_cutout=time_before_cutout, cutout_duration=cutout_duration,
                                   time_after_cutout=time_after_cutout)

run["plot1"].upload(plot1)
run["plot2"].upload(plot2)
run["plot3"].upload(plot3)

torch.save(pred_model, 'model_checkpoints/MSE_model.pt')
run["model_checkpoints/MSE_model.pt"].upload("model_checkpoints/MSE_model.pt")
run.stop()

## Calculating the 95th percentile for prediction MSE

In [ ]:
# Load trained model, and initialize the benchmark model
run = neptune.init_run(with_id="SLEEP-421", mode="read-only",
                       project="sleep-time-series/sleep-time-series",
                       api_token = "--omitted--")
run["model_checkpoints/MSE_model.pt"].download(destination="model_checkpoints/")
benchmark_model = linear_benchmark.BenchmarkModel()
pred_model = torch.load("model_checkpoints/MSE_model.pt")

In [ ]:

# Move to GPU
pred_model.to(device)
benchmark_model.to(device)

# List to save MSE Loss for each section
dataset = individual_validation_datasets[0]
dataset = [dataset[i] for i in range(len(dataset)-1)] # Exclude the last section
MSE_pred_model = np.empty(len(dataset))
MSE_benchmark = np.empty(len(dataset))

# Loop over all sections for subject 3 and save the loss for each section
for i, (inputs, targets) in enumerate(dataset):
    if i % 5000 == 0:
        print(f'Now at section {i}')
    (inputs, targets) = ((
                torch.Tensor(inputs[0]).unsqueeze(0).to(device),
                torch.Tensor(inputs[1]).unsqueeze(0).to(device)),
                (torch.Tensor(targets[0]).unsqueeze(0).to(device),
                torch.Tensor(targets[1]).unsqueeze(0).to(device)
                ))
    output_pred_model = pred_model(inputs, targets, 0)
    output_benchmark = benchmark_model(inputs, targets).to(device)
    loss_pred_model = loss_fn(output_pred_model, targets[0])
    loss_benchmark = loss_fn(output_benchmark, targets[0])
    
    MSE_pred_model[i] = loss_pred_model
    MSE_benchmark[i] = loss_benchmark

# Get the 0.95 quantile from the MSE lists
pred_quantile_95 = np.quantile(MSE_pred_model, 0.95)
benchmark_quentila_95 = np.quantile(MSE_benchmark, 0.95)

Plot a histogram of the prediction MSE for the seq2seq model and the benchmark model

In [ ]:
plt.hist(MSE_pred_model, density = True, fc = (0.3, 0.5, 1, 1), label = "seq2seq", bins = 50, log = True)
plt.hist(MSE_benchmark, density = True, fc = (1.0, 0.0, 0.0, 0.5), label = "Benchmark", bins = 50, log = True)
plt.legend()
plt.xlabel("MSE")
plt.title("Histogram of MSE for seq2seq vs benchmark predictions")

## Anomaly Detection

Now that we have a simple rule for detecting anomalies, try to predict the cutouts for all sections of subject 008

In [ ]:
# create dataset on which to perform anomaly detection
subject = ["008"]
_, individual_anomaly_datasets, _ = anomaly_dataset.create_combined_dataset(subjects=subject,
                                                                          proportion_to_use=1, 
                                                                          time_before_cutout=time_before_cutout,
                                                                          cutout_duration=cutout_duration,
                                                                          original_freq=original_freq, 
                                                                          resample_freq=resample_freq,
                                                                          standardization_info=standardization_info)
anomaly_dataset = individual_anomaly_datasets[0]

# List to save MSE Loss for each section
anomaly_dataset = [anomaly_dataset[i] for i in range(len(anomaly_dataset)-1)] # Exclude the last section
MSE_pred_model = np.empty(len(anomaly_dataset))
MSE_benchmark = np.empty(len(anomaly_dataset))
anomaly = np.empty(len(anomaly_dataset))

# Loop over all sections for subject 3 and save the loss for each section
for i, (inputs, targets, anomaly_bool) in enumerate(anomaly_dataset):
    if i % 5000 == 0:
        print(f"Now at section {i}")
    (inputs, targets) = ((
                torch.Tensor(inputs[0]).unsqueeze(0).to(device),
                torch.Tensor(inputs[1]).unsqueeze(0).to(device)),
                (torch.Tensor(targets[0]).unsqueeze(0).to(device),
                torch.Tensor(targets[1]).unsqueeze(0).to(device)
                ))
    output_pred_model = pred_model(inputs, targets, initial_teacher_forcing)
    output_benchmark = benchmark_model(inputs, targets).to(device)
    loss_pred_model = loss_fn(output_pred_model, targets[0])
    loss_benchmark = loss_fn(output_benchmark, targets[0])
    
    MSE_pred_model[i] = loss_pred_model
    MSE_benchmark[i] = loss_benchmark
    anomaly[i] = anomaly_bool
    
anomaly_pred_model = MSE_pred_model > pred_quantile_95
anomaly_benchmark = MSE_benchmark > benchmark_quentila_95

pred_model_acc = np.sum(anomaly_pred_model == anomaly)/len(anomaly_dataset)
benchmark_acc = np.sum(anomaly_benchmark == anomaly)/len(anomaly_dataset)


In [ ]:
confusion_matrix(anomaly, anomaly_pred_model)

In [ ]:
confusion_matrix(anomaly, anomaly_benchmark)